In [0]:
%run ./utils_dq

In [0]:
dbutils.widgets.text("run_date", "", "Run Date (YYYY-MM-DD)")

In [0]:
run_date = dbutils.widgets.get("run_date")

BRONZE = "abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze"

src_path = f"{BRONZE}/order_items/ingest_date={run_date}"

# Verify the folder exists
display(dbutils.fs.ls(src_path))

# Read only this batch
items_raw = spark.read.parquet(src_path)

print("Bronze batch rows:", items_raw.count())

path,name,size,modificationTime
abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze/order_items/ingest_date=2026-11-01/data_4a4fff06-0bf9-41bc-8ba1-b55c1524f5ee_55fdebe4-a709-4101-8164-7eb4e9a20233.parquet,data_4a4fff06-0bf9-41bc-8ba1-b55c1524f5ee_55fdebe4-a709-4101-8164-7eb4e9a20233.parquet,2007,1793505718000


Bronze batch rows: 50


In [0]:

from pyspark.sql import functions as F

BRONZE = "abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze"

src_path = f"{BRONZE}/order_items/ingest_date={run_date}"

df_items = spark.read.parquet(src_path)

# Check schema
df_items.printSchema()

# Row counts and key uniqueness
display(
    df_items.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("order_item_id").alias("unique_item_ids"),
        F.sum(
            F.col("order_item_id").isNull().cast("int")
        ).alias("null_item_ids")
    )
)

# Inspect sample records
display(df_items.limit(10))


root
 |-- order_item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- item_price: decimal(10,2) (nullable = true)
 |-- created_at: timestamp (nullable = true)



total_rows,unique_item_ids,null_item_ids
50,50,0


order_item_id,order_id,product_id,quantity,item_price,created_at
23266,10041,17,2,999.00,2026-10-31T17:40:19.000Z
23267,10042,9,3,7999.00,2026-10-31T17:40:19.000Z
23268,10042,32,2,799.00,2026-10-31T17:40:19.000Z
23269,10042,35,2,399.00,2026-10-31T17:40:19.000Z
23270,10042,20,2,699.00,2026-10-31T17:40:19.000Z
23271,10043,10,3,6999.00,2026-10-31T17:40:19.000Z
23272,10043,11,3,899.00,2026-10-31T17:40:19.000Z
23273,10044,15,1,549.00,2026-10-31T17:40:19.000Z
23274,10044,2,2,3299.00,2026-10-31T17:40:19.000Z
23275,10045,20,3,699.00,2026-10-31T17:40:19.000Z


In [0]:

from pyspark.sql import functions as F
from delta.tables import DeltaTable

items_clean = (
    df_items
    .filter(F.col("order_item_id").isNotNull())
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("item_price", F.col("item_price").cast("decimal(12,2)"))
    .dropDuplicates(["order_item_id"])
    .drop("ingest_date")
)




In [0]:

# Validate the actual incoming data, without hardcoded counts

source_valid_ids = df_items.filter(
    F.col("order_item_id").isNotNull()
)

expected_unique_ids = (
    source_valid_ids
    .select("order_item_id")
    .distinct()
    .count()
)

clean_count = items_clean.count()

invalid_rows = items_clean.filter(
    F.col("order_id").isNull() |
    F.col("product_id").isNull() |
    F.col("quantity").isNull() |
    (F.col("quantity") <= 0) |
    F.col("item_price").isNull() |
    (F.col("item_price") <= 0)
).count()

print("Cleaned items:", clean_count)
print("Expected unique IDs:", expected_unique_ids)
print("Invalid rows:", invalid_rows)

assert invalid_rows == 0, "Invalid order item records found"
assert clean_count == expected_unique_ids, "Item ID count mismatch"

print("Order Items validation PASSED!")


Cleaned items: 50
Expected unique IDs: 50
Invalid rows: 0
Order Items validation PASSED!


In [0]:

# =====================================================
# STEP 4: CREATE EMPTY SILVER DELTA TABLE (IF NEEDED)
# =====================================================

TARGET = "atliq_dl.silver.order_items"

(
    DeltaTable.createIfNotExists(spark)
    .tableName(TARGET)
    .addColumns(items_clean.schema)
    .execute()
)


# =====================================================
# STEP 5: INSERT-ONLY DELTA MERGE
# =====================================================

(
    DeltaTable.forName(spark, TARGET)
    .alias("t")
    .merge(
        items_clean.alias("s"),
        "t.order_item_id = s.order_item_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Order Items MERGE completed!")


# =====================================================
# STEP 6: VERIFY SILVER TABLE
# =====================================================

silver_items = spark.table(TARGET)

display(
    silver_items.agg(
        F.count("*").alias("silver_rows"),
        F.countDistinct("order_item_id").alias("unique_item_ids")
    )
)


Order Items MERGE completed!


silver_rows,unique_item_ids
23315,23315


In [0]:

from pyspark.sql import functions as F

# 1. Read only the current incremental Bronze batch
run_date = dbutils.widgets.get("run_date")

raw = spark.read.parquet(
    f"{BRONZE}/order_items/ingest_date={run_date}"
)

print("DQ batch rows:", raw.count())

# 2. Count duplicate order_item_ids
duplicate_ids = (
    raw.count()
    - raw.dropDuplicates(["order_item_id"]).count()
)

# 3. Count rows with invalid quantity or price
invalid_rows = raw.filter(
    F.expr("try_cast(quantity AS INT)").isNull() |
    (F.expr("try_cast(quantity AS INT)") <= 0) |
    F.expr("try_cast(item_price AS DECIMAL(12,2))").isNull() |
    (F.expr("try_cast(item_price AS DECIMAL(12,2))") <= 0)
).count()

# 4. Count business duplicate lines (retained in Silver)
business_columns = [
    "order_id", "product_id", "quantity",
    "item_price", "created_at"
]

business_duplicates = (
    raw.count()
    - raw.dropDuplicates(business_columns).count()
)

# 5. Write DQ metrics using our shared utility
log_dq("order_items", "duplicate_item_ids_removed", duplicate_ids)

log_dq("order_items", "invalid_quantity_or_price_found", invalid_rows)

log_dq("order_items", "business_duplicate_lines_retained", business_duplicates)

# 6. Verify DQ history
display(
    spark.table("atliq_dl.silver.dq_log")
    .filter(F.col("table_name") == "order_items")
    .orderBy("run_date", "rule")
)


DQ batch rows: 50


run_date,table_name,rule,rows_affected
2026-10-31,order_items,business_duplicate_lines_retained,101
2026-10-31,order_items,duplicate_item_ids_removed,0
2026-10-31,order_items,invalid_quantity_or_price_found,0
2026-11-01,order_items,business_duplicate_lines_retained,0
2026-11-01,order_items,duplicate_item_ids_removed,0
2026-11-01,order_items,invalid_quantity_or_price_found,0
